# Practical 02(a) — Dimensionality Reduction using PCA

We compare classification performance and training time before and after PCA.

## 1. Import libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


## 2. Load high-dimensional dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("Original feature count:", X.shape[1])


## 3. Split and standardize without leakage

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Baseline model with all features

In [ ]:
baseline = LogisticRegression(max_iter=3000)
start = time.perf_counter()
baseline.fit(X_train_scaled, y_train)
baseline_time = time.perf_counter() - start
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test_scaled))
print("Baseline accuracy:", baseline_accuracy)
print("Training time (seconds):", round(baseline_time, 5))


## 5. Find components that retain at least 95% variance

In [ ]:
pca_all = PCA().fit(X_train_scaled)
cumulative = np.cumsum(pca_all.explained_variance_ratio_)
n_components = np.argmax(cumulative >= 0.95) + 1
print("Components for 95% variance:", n_components)
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.axhline(0.95, linestyle="--")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.grid(True)
plt.show()


## 6. Train reduced model and compare

In [ ]:
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
reduced = LogisticRegression(max_iter=3000)
start = time.perf_counter()
reduced.fit(X_train_pca, y_train)
reduced_time = time.perf_counter() - start
reduced_accuracy = accuracy_score(y_test, reduced.predict(X_test_pca))
pd.DataFrame({
    "Model": ["All features", "PCA reduced"],
    "Feature count": [X_train_scaled.shape[1], X_train_pca.shape[1]],
    "Accuracy": [baseline_accuracy, reduced_accuracy],
    "Training time (s)": [baseline_time, reduced_time]
})
